# Phase A — KAN Variant Sweep

Train 5 KAN variants sequentially and compare training loss.

| Variant | Architecture |
|---------|--------------|
| **A** | B-spline + SiLU base (control) |
| **B** | B-spline + SCReLU base |
| **C** | B-spline, no base path (pure spline) |
| **D** | B-spline + linear base |
| **E** | ReLU-KAN pure basis (arXiv 2406.02075) |

Shared: `768 -> ft(128) CReLU -> KAN(256 -> 128) -> KAN(128 -> 1)`, 40 superbatches, batch 16384, AdamW, StepLR, test77 binpack.

All variants run **unfused** (the `FuseKanLayer` pass is disabled during Phase A).
Expect ~10 min / variant on T4 → ~50 min total.

**Runtime**: Runtime > Change runtime type > GPU (T4 or better).

## 1. Install Rust + clone repo

In [ ]:
%%bash
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    echo 'source $HOME/.cargo/env' >> ~/.bashrc
fi
source $HOME/.cargo/env
rustc --version
cargo --version

In [ ]:
%%bash
set -e
if [ -d /content/bullet ]; then
    cd /content/bullet
    git fetch origin
    git reset --hard origin/main
else
    cd /content
    git clone https://github.com/y0sif/bullet.git
    cd bullet
fi
git log -1 --oneline

## 2. Download training data (test77 binpack)

In [ ]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test77.binpack ]; then
    echo "Downloading test77 binpack from HuggingFace (~1.3 GB compressed)..."
    wget -q -O test77.binpack.zst \
        "https://huggingface.co/datasets/linrock/test77/resolve/main/test77-2022-01-jan-2tb7p.binpack.zst"
    echo "Download complete. Decompressing..."
    zstd -d test77.binpack.zst -o test77.binpack --rm
    echo "Done!"
fi

ls -lh test77.binpack


## 3. Sanity-check: GPU + nvcc

In [ ]:
%%bash
nvidia-smi || echo "WARNING: No GPU detected."
echo "---"
nvcc --version || echo "WARNING: nvcc not found.

## 4. Build all 5 variants (release)

The first build compiles the whole workspace + CUDA kernels (~5 min). The other four are fast.

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet
for v in a b c d e; do
    echo "=== Building kan_variant_${v} ==="
    cargo build --release --example kan_variant_${v} 2>&1 | tail -3
done

## 5. Train all 5 variants sequentially

Each run = 40 superbatches × ~488 batches, logs to `/content/variant_<v>_log.txt`.
If a run fails partway, earlier log files remain — you can re-run this cell starting from the failed variant by editing `VARIANTS`.

In [ ]:
import subprocess, sys, os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

VARIANTS = ["a", "b", "c", "d", "e"]

for v in VARIANTS:
    log_path = f"/content/variant_{v}_log.txt"
    print(f"\n{'='*70}\n Training kan_variant_{v}  ->  {log_path}\n{'='*70}")
    cmd = ["cargo", "run", "--release", "--example", f"kan_variant_{v}"]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        cwd="/content/bullet", text=True, bufsize=1,
    )
    with open(log_path, "w") as log:
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            log.write(line)
    proc.wait()
    print(f"\nVariant {v} exit code: {proc.returncode}")
    if proc.returncode != 0:
        print(f"FAILED — stopping sweep. Fix variant {v} before continuing.")
        break

## 6. Compare loss curves + rank

In [ ]:
import re
import matplotlib.pyplot as plt

def strip_ansi(s):
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

def parse_bullet_log(path):
    losses = []
    with open(path) as f:
        for line in f:
            m = re.search(r'superbatch\s+(\d+)\s+\|.*?running loss\s+([\d.]+)', strip_ansi(line))
            if m:
                losses.append((int(m.group(1)), float(m.group(2))))
    return losses

VARIANT_LABELS = {
    "a": "A: B-spline + SiLU (control)",
    "b": "B: B-spline + SCReLU",
    "c": "C: B-spline (no base)",
    "d": "D: B-spline + linear",
    "e": "E: ReLU-KAN",
}

all_losses = {}
for v in ["a", "b", "c", "d", "e"]:
    path = f"/content/variant_{v}_log.txt"
    try:
        losses = parse_bullet_log(path)
    except FileNotFoundError:
        print(f"Skipping variant {v}: no log at {path}")
        continue
    if losses:
        all_losses[v] = losses
    else:
        print(f"Variant {v}: log exists but no superbatch lines parsed")

if all_losses:
    fig, ax = plt.subplots(figsize=(11, 7))
    for v, losses in all_losses.items():
        ax.plot([x[0] for x in losses], [x[1] for x in losses],
                label=VARIANT_LABELS[v], linewidth=2)
    ax.set_xlabel("Superbatch")
    ax.set_ylabel("Running loss")
    ax.set_title("Phase A — KAN Variant Sweep (40 superbatches each)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/content/phase_a_sweep.png", dpi=150)
    plt.show()

    print("\nFinal loss (lower = better):")
    ranked = sorted(all_losses.items(), key=lambda kv: kv[1][-1][1])
    for rank, (v, losses) in enumerate(ranked, 1):
        marker = " <-- best" if rank == 1 else ""
        print(f"  {rank}. {VARIANT_LABELS[v]:<38}  {losses[-1][1]:.6f}{marker}")
else:
    print("No variant logs parsed.")

## 7. Save results (optional)

Copy logs + plot to Drive for later analysis.

In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/kanue/phase_a'
os.makedirs(dest, exist_ok=True)

for v in ["a", "b", "c", "d", "e"]:
    src = f"/content/variant_{v}_log.txt"
    if os.path.exists(src):
        shutil.copy(src, dest)

if os.path.exists('/content/phase_a_sweep.png'):
    shutil.copy('/content/phase_a_sweep.png', dest)

# Save checkpoints if any variant produced them
ckpt_root = '/content/bullet/checkpoints'
if os.path.isdir(ckpt_root):
    for d in os.listdir(ckpt_root):
        if d.startswith('kan-variant-'):
            src = os.path.join(ckpt_root, d)
            shutil.copytree(src, os.path.join(dest, 'checkpoints', d), dirs_exist_ok=True)

print(f"Saved to: {dest}")